In [ ]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

def html_to_text_with_csv_tables(html):
    soup = BeautifulSoup(html, 'html.parser')
    main = soup.find('main') or soup

    text_lines = []
    table_count = 0

    for elem in main.descendants:
        if elem.name in ['h1', 'h2', 'h3', 'h4', 'h5']:
            text_lines.append('\n' + elem.get_text(strip=True).upper() + '\n' + ('=' * 40))
        elif elem.name == 'p':
            ptext = elem.get_text(strip=True)
            if ptext:
                text_lines.append(ptext)
        elif elem.name == 'table':
            for tr in elem.find_all('tr'):
                row = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
                text_lines.append(','.join(row))

    return clean_text('\n\n'.join(text_lines))


In [ ]:
from neo4j import GraphDatabase
import json

with open('neo4j_dbinfo', 'r') as f:
    neo4j_info = json.load(f)

uri = neo4j_info["uri"]
user = neo4j_info["username"]
password = neo4j_info["password"]

driver = GraphDatabase.driver(uri, auth=(user, password))

def get_nodes(tx, label):
    query = f"MATCH (n:{label}) RETURN n.id AS id"
    result = tx.run(query)
    return [record["id"] for record in result]

def update_node_text(tx, label, node_id, text):
    query = f"""
    MATCH (n:{label} {{id: $id}})
    SET n.text_content = $text
    """
    tx.run(query, id=node_id, text=text)

def load_html_file(label, node_id):
    base_paths = {
        "AOP": "aops_html",
        "Event": "events_html",
        "KE_Relation": "relationships_html",
        "Stressor": "stressors_html"
    }
    base_path = base_paths.get(label)
    if not base_path:
        return None
    filepath = f"{base_path}/{node_id}/{node_id}.html"
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return html_to_text_with_csv_tables(f.read())
    except FileNotFoundError:
        print(f"[WARN] File not found: {filepath}")
        return None


In [ ]:
def main():
    with driver.session() as session:
        labels = ["AOP", "Event", "KE_Relation", "Stressor"]
        for label in labels:
            print(f"Processing label: {label}")
            node_ids = session.read_transaction(get_nodes, label)
            for node_id in node_ids:
                html_text = load_html_file(label, node_id)
                if html_text:
                    session.write_transaction(update_node_text, label, node_id, html_text)
                    print(f"Updated node {label} {node_id}")
                else:
                    print(f"Skipped node {label} {node_id} (no html)")

if __name__ == "__main__":
    main()